# SWOT WSE Acquisition — Canal Baker

Searches SWOT L2_HR_Raster granules via earthaccess, samples WSE at 22
mid-channel points via direct S3 access (no download), and writes results
to S3 parquet.

**Output:** `s3://esip-qhub/canal-baker-tides/swot_wse_samples.parquet`

**Run on:** Coiled notebook (`coiled notebook start --name tides --software tides --disk-size 50`)

In [ ]:
import earthaccess
import xarray as xr
import pandas as pd
import numpy as np
import s3fs

# Verify short_name against Task 2 findings
SWOT_SHORT_NAME = "SWOT_L2_HR_Raster_2.0"
BBOX = (-75.38, -48.74, -73.23, -47.12)   # (W, S, E, N)
TEMPORAL = ("2023-07-01", None)
OUTPUT_PATH = "s3://esip-qhub/canal-baker-tides/swot_wse_samples.parquet"

GOOD_QUAL = {0, 1}           # 0=good, 1=suspect; discard 2=degraded, 3=bad
WATER_FRAC_THRESHOLD = 0.9

In [ ]:
earthaccess.login()

points = pd.read_csv("sample_points.csv")
print(f"Loaded {len(points)} sample points")
display(points)

In [ ]:
results = earthaccess.search_data(
    short_name=SWOT_SHORT_NAME,
    bounding_box=BBOX,
    temporal=TEMPORAL,
)
print(f"Found {len(results)} granules")

In [ ]:
def sample_granule(ds, points):
    """
    Extract WSE at each sample point from one SWOT L2_HR_Raster granule.
    Uses nearest-pixel lookup on the 2D lat/lon grid.
    Returns list of dicts; skips points with no valid pixel.
    """
    lat_grid = ds["latitude"].values   # 2D array (y, x) — update name if Task 2 found different
    lon_grid = ds["longitude"].values  # 2D array (y, x)

    # Extract overpass timestamp from granule
    if "time" in ds:
        timestamp = pd.Timestamp(ds["time"].values.flat[0])
    elif "time_tai" in ds:
        timestamp = pd.Timestamp(ds["time_tai"].values.flat[0])
    else:
        timestamp = None  # update variable name from Task 2 findings

    rows = []
    for _, pt in points.iterrows():
        dist = (lat_grid - pt["lat"]) ** 2 + (lon_grid - pt["lon"]) ** 2
        if np.all(np.isnan(dist)):
            continue
        idx = np.unravel_index(np.nanargmin(dist), dist.shape)

        wse = float(ds["wse"].values[idx])
        wse_qual = int(ds["wse_qual"].values[idx])
        water_frac = float(ds["water_frac"].values[idx])
        cross_track = float(ds["cross_track"].values[idx]) if "cross_track" in ds else np.nan

        if wse_qual in GOOD_QUAL and water_frac > WATER_FRAC_THRESHOLD and not np.isnan(wse):
            rows.append({
                "point_id": pt["id"],
                "lat": pt["lat"],
                "lon": pt["lon"],
                "timestamp": timestamp,
                "wse": wse,
                "wse_qual": wse_qual,
                "water_frac": water_frac,
                "cross_track": cross_track,
            })
    return rows

In [ ]:
test_files = earthaccess.open([results[0]])
test_ds = xr.open_dataset(test_files[0], engine='h5netcdf')
test_rows = sample_granule(test_ds, points)
print(f"Valid observations from test granule: {len(test_rows)}")
if test_rows:
    display(pd.DataFrame(test_rows))

In [ ]:
all_rows = []

for i, granule in enumerate(results):
    if i % 50 == 0:
        print(f"  {i+1}/{len(results)} granules — {len(all_rows)} observations so far")
    try:
        files = earthaccess.open([granule])
        ds = xr.open_dataset(files[0], engine='h5netcdf')
        rows = sample_granule(ds, points)
        all_rows.extend(rows)
        ds.close()
    except Exception as e:
        print(f"  Skipped granule {i}: {e}")

print(f"\nTotal valid observations: {len(all_rows)}")

In [ ]:
df = pd.DataFrame(all_rows)
df["timestamp"] = pd.to_datetime(df["timestamp"], utc=True)
df = df.sort_values(["point_id", "timestamp"]).reset_index(drop=True)

fs = s3fs.S3FileSystem()
with fs.open(OUTPUT_PATH, "wb") as f:
    df.to_parquet(f, index=False)

print(f"Wrote {len(df)} rows to {OUTPUT_PATH}")
print(f"Date range: {df['timestamp'].min()} \u2192 {df['timestamp'].max()}")

In [ ]:
obs_per_point = (
    df.groupby("point_id").size()
    .reindex(points["id"])
    .fillna(0)
    .astype(int)
    .rename("n_obs")
)
display(obs_per_point.to_frame())

zero = obs_per_point[obs_per_point == 0]
if len(zero):
    print(f"WARNING: {len(zero)} points with zero observations: {list(zero.index)}")
    print("These points likely fall in the SWOT nadir gap for both pass geometries.")
else:
    print("All 22 points have at least one observation.")